In [ ]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time          

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기 
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xls형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xls): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text()
time.sleep(1)
print('검색하신 키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ]        #번호 저장
title2 = [ ]     #논문제목 저장
writer2 = [ ]    #논문저자 저장
org2 = [ ]       #소속기관 저장
no = 1

for a in range(1, collect_page_cnt + 1) :
    
    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')
    
    for b in content_2 :    
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text()
        except :
            continue
        else :
            f = open(ft_name, 'a' , encoding="UTF-8")
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))

            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n' + '2.논문제목:' + title)
            
            writer = b.find('span','writer').get_text()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()
            print('4.소속기관:' , org)
            org2.append(org)
            f.write('\n' + '4.소속기관:' + org + '\n')
            
            f.close( )
            
            no += 1
            print("\n")
            
            if no > collect_cnt :
                break

            time.sleep(1)        # 페이지 변경 전 1초 대기 

    a += 1 
    b = str(a)
    time.sleep(2)

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
        time.sleep(2) 
    except :
        driver.find_element(By.LINK_TEXT, '다음 페이지로').click()
        time.sleep(2)
        
print("요청하신 작업이 모두 완료되었습니다")

# Step 10. 수집된 데이터를 xls와 csv 형태로 저장하기
import pandas as pd 

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name,index=False, encoding="utf-8" , engine='openpyxl')

# csv 형태로 저장하기
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

In [4]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time          

def get_genre():
    genre_dict = {
        "1": "학위논문",
        "2": "국내학술논문",
        "3": "해외학술논문",
        "4": "학술지",
        "5": "단행본",
        "6": "공개강의",
        "7": "연구보고서",
        "8": "학회지",
        "9": "연구논문",
        "10": "학술발표자료",
        "11": "특허",
        "12": "기타"
    }

    information = input(
        "2. 위 키워드로 검색하여 아래의 장르 중 어떤 장르의 정보를 수집할까요?\n"
        "1. 학위논문  2. 국내학술논문  3. 해외학술논문  4. 학술지  5. 단행본\n"
        "6. 공개강의  7. 연구보고서  8. 학회지  9. 연구논문  10. 학술발표자료\n"
        "11. 특허  12. 기타\n"
        "위 장르 중 수집할 장르의 번호를 입력하세요: "
    )

    genre = genre_dict.get(information.strip())  # 입력값 앞뒤 공백 제거 후 매칭

    if genre:
        print(f"선택한 장르: {genre}")
        return genre
    else:
        print("잘못된 입력입니다. 1~12 사이의 숫자를 입력하세요.")
        return get_genre()  # 재귀 호출로 다시 입력받음

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

print("=" *100)
information = get_genre()


#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기 
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xls형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xls): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT, information).click()
time.sleep(5)

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
선택한 장르: 국내학술논문


In [6]:
#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
from urllib.parse import urljoin
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox').find('span','num').get_text()
time.sleep(1)
print('검색하신 키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ]        #번호 저장
title2 = [ ]     #논문제목 저장
writer2 = [ ]    #논문저자 저장
org2 = [ ]       #소속기관 저장
release2 = [ ]   #발표년도 저장
paper2 = [ ]     #논문집/자료집 저장
url2 = [ ]       #논문 url 정보 저장

no = 1

for a in range(1, collect_page_cnt + 1) :
    
    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')
    
    for b in content_2 :    
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text()
        except :
            continue
        else :
            print('1.번호:',no)
            no2.append(no)

            print('2.논문제목:',title)
            title2.append(title)
            
            writer = b.find('span','writer').get_text()
            print('3.저자:',writer)
            writer2.append(writer)

            org = b.find('span','assigned').get_text()
            print('4.소속기관:' , org)
            org2.append(org)

            release = b.find('p','etc').find_all('span')[2].get_text()
            print('5.발표년도:',release)
            release2.append(release)

            try:
                paper = b.find('p', 'preAbstract').get_text()
                print('6.논문집/자료집:', paper)
                paper2.append(paper)
            except:
                paper2.append('')  # 데이터가 없을 경우 빈 문자열 추가

            #  논문 URL을 상대 경로에서 절대 경로로 변환
            relative_url = b.find('p', class_='title').find('a')['href']
            base_url = driver.current_url  # 현재 페이지의 URL 가져오기
            absolute_url = urljoin(base_url, relative_url)

            print('7.논문 URL:', absolute_url)
            url2.append(absolute_url)

            no += 1
            print("\n")
            
            if no > collect_cnt :
                break

            time.sleep(1)        # 페이지 변경 전 1초 대기 

    a += 1 
    b = str(a)
    time.sleep(2)

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
        time.sleep(2) 
    except :
        driver.find_element(By.LINK_TEXT, '다음 페이지로').click()
        time.sleep(2)
        
print("요청하신 작업이 모두 완료되었습니다")

# Step 10. 수집된 데이터를 xls와 csv 형태로 저장하기
import pandas as pd 

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)
df['발표년도']=pd.Series(release2)
df['논문집/자료집']=pd.Series(paper2)
df['논문 URL']=pd.Series(url2)

# xls 형태로 저장하기
df.to_excel(fx_name,index=False, engine='openpyxl')

# csv 형태로 저장하기
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

1메타버스 시대 현대형법의 한계와 미래형법의 마련허재은(Jae-Eun Heo),이경렬(Kyung-Lyul Lee)한국형사정책학회2022刑事政策Vol.34 No.3원문보기 2KCI스콜라구독기관에 따라 유료논문이 존재할 수 있습니다.메타버스는 기존의 온라인 공간에서 이루어지는 채팅이나 게임 등과는 구별이 되는 개념으로, 사용자가 가상세계에서 아바타라는 존재를 통해 현실과 거의 유사한사회·경제·문화 활동을 할 수 있다. 사용자는 아바타를 통해 현실과 같은 의사 소통이 가능하고, 메타버스라는 공간에서 사용자 자신의 창작물을 만들어 낼 수 있으며, 특정 아이템을 매매하는 등의 거래도 가능하다. 메타버스는 가상의 공간이라는 제약이 덜하고 아바타를 통해 메타버스 내의 독립적인 캐릭터를 구축할 수 있으며, 사용자의 현실 세계와는 완전히 다른 독보적인 활동이 가능하게 되었다.그러나 이러한 메타버스에서의 활동이 증가하면서 범법적 행위 역시 증가하고 있는데, 기존의 범죄 형태와 유사한 부분도 있고 완전히 다른 부분도 있다. 메타버스에서는 아바타라는 존재가 사용자를 대신하여 모든 활동을 수행하고 있는데, 사용자와별개의 행위 주체 및 객체로 인정할 수 있는지 여부에 따라 기존 형사법의 적용도 완전히 달라 질 수 있다.메타버스 내에서 명예에 관한 범죄가 발생하였을 경우 과연 피해자를 특정할 수있을 것인지, 사용자와 아바타 사이에 사회적으로 인정받는 가치가 다를 경우 어떻게 적용할 것인지 등에 대하여 검토되어야 한다. 메타버스 공간에서 발생의 빈도가늘어가고 있는 성범죄와 관련하여, 직접적인 신체적 침해는 없음에도 피해자가 받는정신적인 피해는 동일한 경우 현행 형사법으로 처벌할 수 있는지 살펴볼 것이다. 그외 메타버스에서 재화에 대한 교환 및 매매가 활발히 이루어지고 있는데 절취 또는 편취의 행위가 발생할 경우, 이를 기존의 형사법에 그대로 대입하여 처벌이 가능한지도 살펴볼 것이다.메타버스 시대에 기존의 형사법으로 해결할 수 없는 부분은 분명히 등장할 것이고 이를 대비하는 입법적 활동이 